In [1]:
!pip install optuna mlflow dagshub seaborn lightgbm

In [2]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import lightgbm as lgb
import dagshub
import mlflow
import mlflow.lightgbm

In [3]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [4]:
df = pd.read_csv('/content/twitter_cleaned.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51618 entries, 0 to 51617
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  51618 non-null  object
 1   text       51615 non-null  object
dtypes: object(2)
memory usage: 806.7+ KB


In [6]:
df.dropna(inplace=True)

In [7]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [8]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [9]:
import spacy

In [10]:
nlp = spacy.load('en_core_web_sm')

In [11]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [12]:
import pandas as pd
import numpy as np
import spacy

# Load spaCy model once (outside function for efficiency)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def extract_text_features(text_series):
    """
    Extract NLP statistical + POS-based features from text.

    Parameters:
        text_series (pd.Series): Series of raw text

    Returns:
        pd.DataFrame: Feature dataframe
    """

    features = []

    for doc in nlp.pipe(text_series.astype(str), batch_size=64):

        text = doc.text
        tokens = [token for token in doc if not token.is_space]
        words = [token.text for token in tokens if token.is_alpha]

        word_lengths = [len(word) for word in words]

        # Basic statistics
        comment_length = len(text)
        word_count = len(words)
        unique_word_count = len(set(words))
        avg_word_length = np.mean(word_lengths) if word_lengths else 0
        lexical_diversity = (
            unique_word_count / word_count if word_count > 0 else 0
        )

        # POS counts
        pos_counts = {}
        for token in tokens:
            pos_counts[token.pos_] = pos_counts.get(token.pos_, 0) + 1

        total_tokens = len(tokens)

        # POS proportions
        pos_proportions = {
            f"prop_{pos}": count / total_tokens
            for pos, count in pos_counts.items()
        } if total_tokens > 0 else {}

        row = {
            "comment_length": comment_length,
            "word_count": word_count,
            "avg_word_length": avg_word_length,
            "unique_word_count": unique_word_count,
            "lexical_diversity": lexical_diversity,
        }

        # Add POS counts
        for pos, count in pos_counts.items():
            row[f"count_{pos}"] = count

        # Add POS proportions
        row.update(pos_proportions)

        features.append(row)

    features_df = pd.DataFrame(features).fillna(0)

    return features_df

In [13]:
"""
import re
import numpy as np
import pandas as pd
import spacy
from nltk.sentiment import SentimentIntensityAnalyzer

# Load once
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
sia = SentimentIntensityAnalyzer()

NEGATIONS = {"not","no","never","none","nobody","nothing","neither","nowhere","hardly","scarcely"}
INTENSIFIERS = {"very","extremely","really","absolutely","totally","highly","so"}
DIMINISHERS = {"slightly","somewhat","little","barely","hardly"}
CONTRAST_MARKERS = {"but","however","although","though"}

def extract_all_features(text_series):

    all_rows = []

    for doc in nlp.pipe(text_series.astype(str), batch_size=64):

        text = doc.text
        tokens = [token for token in doc if not token.is_space]
        words = [token.text for token in tokens if token.is_alpha]

        word_count = len(words)
        word_lengths = [len(w) for w in words]

        # BASIC STATISTICAL FEATURES

        comment_length = len(text)
        unique_word_count = len(set(words))
        avg_word_length = np.mean(word_lengths) if word_lengths else 0
        lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

        # POS FEATURES

        pos_counts = {}
        for token in tokens:
            pos_counts[token.pos_] = pos_counts.get(token.pos_, 0) + 1

        total_tokens = len(tokens)

        pos_proportions = {
            f"prop_{pos}": count / total_tokens
            for pos, count in pos_counts.items()
        } if total_tokens > 0 else {}

        # SENTIMENT FEATURES

        # VADER
        vader = sia.polarity_scores(text)

        # Negation
        negation_count = sum(w.lower() in NEGATIONS for w in words)
        negation_ratio = negation_count / word_count if word_count > 0 else 0

        # Intensifiers / Diminishers
        intensifier_count = sum(w.lower() in INTENSIFIERS for w in words)
        diminisher_count = sum(w.lower() in DIMINISHERS for w in words)

        # Punctuation
        exclamation_count = text.count("!")
        question_count = text.count("?")

        # Uppercase emphasis
        uppercase_words = [w for w in words if w.isupper() and len(w) > 1]
        uppercase_count = len(uppercase_words)
        uppercase_ratio = uppercase_count / word_count if word_count > 0 else 0

        # Elongation
        elongated_count = len(re.findall(r"(.)\1{2,}", text))

        # Contrast markers
        contrast_count = sum(w.lower() in CONTRAST_MARKERS for w in words)

        # MERGE ALL FEATURES

        row = {
            # Basic
            "comment_length": comment_length,
            "word_count": word_count,
            "unique_word_count": unique_word_count,
            "avg_word_length": avg_word_length,
            "lexical_diversity": lexical_diversity,

            # VADER
            "vader_compound": vader["compound"],
            "vader_pos": vader["pos"],
            "vader_neg": vader["neg"],
            "vader_neu": vader["neu"],

            # Sentiment mechanics
            "negation_count": negation_count,
            "negation_ratio": negation_ratio,
            "intensifier_count": intensifier_count,
            "diminisher_count": diminisher_count,
            "exclamation_count": exclamation_count,
            "question_count": question_count,
            "uppercase_count": uppercase_count,
            "uppercase_ratio": uppercase_ratio,
            "elongated_count": elongated_count,
            "contrast_count": contrast_count,
        }

        # Add POS counts
        for pos, count in pos_counts.items():
            row[f"count_{pos}"] = count

        # Add POS proportions
        row.update(pos_proportions)

        all_rows.append(row)

    return pd.DataFrame(all_rows).fillna(0)
"""

'\nimport re\nimport numpy as np\nimport pandas as pd\nimport spacy\nfrom nltk.sentiment import SentimentIntensityAnalyzer\n\n# Load once\nnlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])\nsia = SentimentIntensityAnalyzer()\n\nNEGATIONS = {"not","no","never","none","nobody","nothing","neither","nowhere","hardly","scarcely"}\nINTENSIFIERS = {"very","extremely","really","absolutely","totally","highly","so"}\nDIMINISHERS = {"slightly","somewhat","little","barely","hardly"}\nCONTRAST_MARKERS = {"but","however","although","though"}\n\ndef extract_all_features(text_series):\n\n    all_rows = []\n\n    for doc in nlp.pipe(text_series.astype(str), batch_size=64):\n\n        text = doc.text\n        tokens = [token for token in doc if not token.is_space]\n        words = [token.text for token in tokens if token.is_alpha]\n\n        word_count = len(words)\n        word_lengths = [len(w) for w in words]\n\n        # BASIC STATISTICAL FEATURES\n\n        comment_length = len(text)\n 

In [14]:
train_custom_features = extract_text_features(X_train_text)
test_custom_features = extract_text_features(X_test_text)

In [15]:
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [16]:
train_custom_features.columns

Index(['comment_length', 'word_count', 'avg_word_length', 'unique_word_count',
       'lexical_diversity', 'count_PROPN', 'count_ADJ', 'count_ADV',
       'count_VERB', 'prop_PROPN', 'prop_ADJ', 'prop_ADV', 'prop_VERB',
       'count_NUM', 'count_NOUN', 'count_X', 'count_INTJ', 'prop_NUM',
       'prop_NOUN', 'prop_X', 'prop_INTJ', 'count_PRON', 'count_ADP',
       'prop_PRON', 'prop_ADP', 'count_PART', 'prop_PART', 'count_DET',
       'prop_DET', 'count_CCONJ', 'count_AUX', 'prop_CCONJ', 'prop_AUX',
       'count_PUNCT', 'prop_PUNCT', 'count_SCONJ', 'prop_SCONJ', 'count_SYM',
       'prop_SYM'],
      dtype='object')

In [17]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train_bow = X_train.astype(np.float32)
X_test_bow = X_test.astype(np.float32)

num_classes = len(np.unique(y_train))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Num classes:", num_classes)

Train shape: (41292, 8000)
Test shape: (10323, 8000)
Num classes: 3


In [18]:
X_train_df = pd.DataFrame(X_train_bow.toarray(), columns=vectorizer.get_feature_names_out())
X_test_df = pd.DataFrame(X_test_bow.toarray(), columns=vectorizer.get_feature_names_out())

In [19]:
X_train_final = pd.concat([X_train_df.reset_index(drop=True), train_custom_features.reset_index(drop=True)], axis=1)
X_test_final = pd.concat([X_test_df.reset_index(drop=True), test_custom_features.reset_index(drop=True)], axis=1)

In [20]:
X_train_final

,00,0016,01,02,03,03 league,04,05,06,07,...,count_CCONJ,count_AUX,prop_CCONJ,prop_AUX,count_PUNCT,prop_PUNCT,count_SCONJ,prop_SCONJ,count_SYM,prop_SYM
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41287,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
41288,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
41289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
41290,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.038462,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
def build_lgb(trial):

    return lgb.LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
        max_depth=trial.suggest_int("max_depth", -1, 15),
        num_leaves=trial.suggest_int("num_leaves", 20, 200),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        objective="multiclass",
        num_class=num_classes,
        random_state=42,
        is_unbalance=True,
        class_weight="balanced",
        n_jobs=-1,
        verbosity=-1
    )

In [22]:
def objective(trial):

    model = build_lgb(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(
            X_train_final,
            y_train,
            callbacks=[lgb.log_evaluation(0)]
        )

        preds = model.predict(X_test_final)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [23]:
study = optuna.create_study(
    direction="maximize",
)

[I 2026-02-24 13:39:11,996] A new study created in memory with name: no-name-7235035e-1fd9-4d57-a99a-eabd8c9a04a8


In [24]:
with mlflow.start_run(run_name="LGBM-with-simple-custom-features"):

    mlflow.log_param("model_name", "LightGBM")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train_final.shape[0])
    mlflow.log_param("num_features", X_train_final.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = lgb.LGBMClassifier(
        **best_params,
        objective="multiclass",
        num_class=num_classes,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    final_model.fit(
        X_train_final,
        y_train,
        callbacks=[lgb.log_evaluation(0)]
    )

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = lgb.LGBMClassifier(
            **best_params,
            objective="multiclass",
            num_class=num_classes,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )

        model.fit(
            X_tr,
            y_tr,
            callbacks=[lgb.log_evaluation(0)]
        )

        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test_final)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_lgb.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_lgb.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - LightGBM")
    plt.savefig("confusion_matrix_lgb.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_lgb.png")

    study.trials_dataframe().to_csv("optuna_trials_lgb.csv", index=False)
    mlflow.log_artifact("optuna_trials_lgb.csv")

    mlflow.lightgbm.log_model(final_model, artifact_path="model")

print("LightGBM experiment completed successfully.")

[I 2026-02-24 13:39:40,479] Trial 0 finished with value: 0.6620313394037985 and parameters: {'n_estimators': 183, 'learning_rate': 0.13266594909311935, 'max_depth': 11, 'num_leaves': 35, 'min_child_samples': 45, 'subsample': 0.7773476069477765, 'colsample_bytree': 0.7846033156038489}. Best is trial 0 with value: 0.6620313394037985.


🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/5f83321d137f4d17ab726219456d7fb5
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-24 13:40:08,987] Trial 1 finished with value: 0.6697518828989343 and parameters: {'n_estimators': 55, 'learning_rate': 0.2204486033321068, 'max_depth': 15, 'num_leaves': 81, 'min_child_samples': 18, 'subsample': 0.9777163423306796, 'colsample_bytree': 0.7656567752406813}. Best is trial 1 with value: 0.6697518828989343.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/990fdbeccc0f4f85a9c0b9d21b3bfa2d
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
🏃 View run trial_2 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/7d39905c7e4f4340bdd07284ca5ee8c0
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-24 13:40:30,777] Trial 2 finished with value: 0.6355164666914396 and parameters: {'n_estimators': 133, 'learning_rate': 0.2369415957616065, 'max_depth': 5, 'num_leaves': 172, 'min_child_samples': 41, 'subsample': 0.7684309297112649, 'colsample_bytree': 0.7433268535467982}. Best is trial 1 with value: 0.6697518828989343.
[I 2026-02-24 13:41:04,317] Trial 3 finished with value: 0.714987408045131 and parameters: {'n_estimators': 298, 'learning_rate': 0.23822075814153446, 'max_depth': 14, 'num_leaves': 43, 'min_child_samples': 39, 'subsample': 0.9068685436444616, 'colsample_bytree': 0.6480807371903778}. Best is trial 3 with value: 0.714987408045131.


...

[I 2026-02-24 14:53:08,583] Trial 48 finished with value: 0.5576318448502092 and parameters: {'n_estimators': 177, 'learning_rate': 0.20058189686675781, 'max_depth': 1, 'num_leaves': 80, 'min_child_samples': 14, 'subsample': 0.8225283907720902, 'colsample_bytree': 0.6591297965630319}. Best is trial 41 with value: 0.8087421136759742.


🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/f9583d37d4784322aa6752f29f941b1b
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-24 14:54:05,423] Trial 49 finished with value: 0.6604609798768161 and parameters: {'n_estimators': 294, 'learning_rate': 0.2241978733701982, 'max_depth': 3, 'num_leaves': 109, 'min_child_samples': 7, 'subsample': 0.9271400726507423, 'colsample_bytree': 0.6765198354644774}. Best is trial 41 with value: 0.8087421136759742.


Best Macro F1: 0.8087421136759742
Best Params: {'n_estimators': 477, 'learning_rate': 0.2820094019106475, 'max_depth': -1, 'num_leaves': 144, 'min_child_samples': 7, 'subsample': 0.841052534213301, 'colsample_bytree': 0.6275132628207281}
🏃 View run LGBM-with-simple-custom-features at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/b1bebd4b71af47e282af76f02c15a149
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


KeyError: "None of [Index([    1,     2,     3,     4,     5,     8,     9,    11,    13,    14,\n       ...\n       41280, 41281, 41283, 41284, 41285, 41286, 41287, 41288, 41290, 41291],\n      dtype='int64', length=33033)] are in the [columns]"